# 🧪 معمل التجارب: بناء محرك التقطيع الدلالي بلغة بايثون (Lecture 38)
أهلاً بك في معملك التطبيقي للمحاضرة 38 من مرحلة التقطيع الدلالي المتقدم (`Advanced Semantic Chunking`)! 🚀

**فكرة المعمل ببساطة:**
في المحاضرة السابقة فهمنا الفكرة الرياضية للتقطيع الدلالي. اليوم سنقوم **ببنائه عملياً بأيدينا من الصفر** باستخدام مكتبة `sentence-transformers` وموديل `all-MiniLM-L6-v2` المجاني والخفيف، مع دالة `cosine_similarity` من `scikit-learn`.

ستتعلم وتختبر في هذا المعمل:
1. استدعاء وتحميل موديل التضمين الخفيف محلياً.
2. قراءة وتقسيم النص إلى جمل مستقلة ونقية.
3. حساب متجهات التضمين الكثيفة للجمل.
4. حساب التشابه بين الجمل المتتالية ورصد الهبوط الحاد في المعنى (`Cliff Drop`).
5. تطبيق خوارزمية التجميع الديناميكي بالعتبة (`Threshold Grouping`).
6. اختبار تأثير العتبة المحكمة (`0.7`) مقابل العتبة المتوازنة (`0.35`).

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز وشغله ⬅️ ثم طبق بنفسك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحميل موديل التضمين
**الفكرة ببساطة:**
بنستورد مكتبة `sentence-transformers` ومكتبة `scikit-learn` لحساب تشابه جيب التمام (`cosine_similarity`).
ونحمل موديل `all-MiniLM-L6-v2` الذي يحول أي جملة لمتجه طوله 384 رقماً بسرعة فائقة ومجاناً.

In [1]:
# 1. import core libraries
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# initialize local embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Model loaded successfully!")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded successfully!
Embedding dimension: 384


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_15192\4016875880.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


In [2]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتحميل الموديل بنفسك هنا وشغله):


### 2. تجهيز النص الخام وتجزئته إلى جمل مستقلة
**الفكرة ببساطة:**
بدل ما نتعامل مع النص ككتلة واحدة، بنقسمه لجمل مفردة وننظف المسافات الزائدة (`strip`).
سنستخدم نفس الجمل الـ 5 التي شرحها المحاضر في الفيديو (3 جمل عن الذكاء الاصطناعي ولانج تشين، وجملتان عن فرنسا وبرج إيفل).

In [3]:
# raw sample text with multi-topic sentences
text = """LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

# split into clean sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

print(f"Total extracted sentences: {len(sentences)}\n")
for i, s in enumerate(sentences, start=1):
    print(f"[{i}] {s}")

Total extracted sentences: 5

[1] LangChain is a framework for building applications with LLMs.
[2] LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
[3] You can create chains, agents, memory, and retrievers.
[4] The Eiffel Tower is located in Paris.
[5] France is a popular tourist destination.


In [4]:
# ✍️ مساحتك للتطبيق العملي (جرب كتابة نص من عندك وقسمه لجمل مستقلة هنا وشغله):


### 3. توليد متجهات التضمين للجمل (`Sentence Embeddings`)
**الفكرة ببساطة:**
بنمرر قائمة الجمل كاملة للموديل عبر `model.encode(sentences)`، لنحصل على مصفوفة متجهات (كل جملة لها 384 بعداً رياضياً يمثل معناها).

In [5]:
# encode all sentences into dense vectors
embeddings = model.encode(sentences)

print(f"Embeddings matrix shape: {embeddings.shape}")
print(f"Sample of sentence 1 vector (first 5 dims): {embeddings[0][:5]}")

Embeddings matrix shape: (5, 384)
Sample of sentence 1 vector (first 5 dims): [-0.02109225 -0.04472177  0.0108708  -0.09759173 -0.00697066]


In [6]:
# ✍️ مساحتك للتطبيق العملي (جرب حساب التضمين لجملك واطبع أبعاد المصفوفة هنا):


### 4. حساب التشابه الزوجي ورصد الهبوط الحاد (`Pairwise Cosine Similarity`)
**الفكرة ببساطة:**
بنقيس زاوية التشابه بين كل جملة والجملة التي بعدها.
ستلاحظ أن التشابه بين الجملة 1 و2 عالي جداً (~0.82)، لكن بين الجملة 3 و4 يحدث **انهيار مفاجئ** (~0.02) لأننا انتقلنا من الحديث عن مكونات الذكاء الاصطناعي إلى برج إيفل!

In [7]:
# compute cosine similarity between adjacent sentences
print("🔍 Pairwise Similarity Scores:\n")
for i in range(len(sentences) - 1):
    sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
    print(f"Sim(S_{i+1}, S_{i+2}) = {sim:.4f}")
    print(f"   S_{i+1}: '{sentences[i][:40]}...'")
    print(f"   S_{i+2}: '{sentences[i+1][:40]}...'")
    if sim < 0.2:
        print("   🚨 Significant drop detected -> Topic change!\n")
    else:
        print("   🔗 High semantic continuity.\n")

🔍 Pairwise Similarity Scores:

Sim(S_1, S_2) = 0.8248
   S_1: 'LangChain is a framework for building ap...'
   S_2: 'LangChain provides modular abstraction t...'
   🔗 High semantic continuity.

Sim(S_2, S_3) = 0.4487
   S_2: 'LangChain provides modular abstraction t...'
   S_3: 'You can create chains, agents, memory, a...'
   🔗 High semantic continuity.

Sim(S_3, S_4) = 0.0225
   S_3: 'You can create chains, agents, memory, a...'
   S_4: 'The Eiffel Tower is located in Paris....'
   🚨 Significant drop detected -> Topic change!

Sim(S_4, S_5) = 0.3776
   S_4: 'The Eiffel Tower is located in Paris....'
   S_5: 'France is a popular tourist destination....'
   🔗 High semantic continuity.



In [8]:
# ✍️ مساحتك للتطبيق العملي (جرب حساب التشابه بين جملك وافحص أين يحدث الهبوط الحاد هنا):


### 5. خوارزمية التجميع والتقطيع بالعتبة (`Threshold = 0.7`)
**الفكرة ببساطة:**
هنمر بحلقة `for`:
- لو التشابه بين الجملة الحالية والسابقة >= العتبة (0.7): ندمجها في نفس الـ Chunk.
- لو التشابه أقل من العتبة: نقفل الـ Chunk القديم، ونبدأ Chunk جديد!
- في النهاية بنضيف الـ Chunk الأخير.

In [9]:
# semantic grouping function
def semantic_chunk_text(sentences, embeddings, threshold=0.7):
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

# run with lecture threshold = 0.7
chunks_tight = semantic_chunk_text(sentences, embeddings, threshold=0.7)

print(f"📦 Chunks with Threshold = 0.7 (Total: {len(chunks_tight)}):\n")
for idx, c in enumerate(chunks_tight, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{c}\n")

📦 Chunks with Threshold = 0.7 (Total: 4):

--- Chunk [1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.

--- Chunk [2] ---
You can create chains, agents, memory, and retrievers.

--- Chunk [3] ---
The Eiffel Tower is located in Paris.

--- Chunk [4] ---
France is a popular tourist destination.



In [10]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل دالة التقطيع بالعتبة 0.7 واطبع النتائج هنا):


### 6. تجربة ضبط العتبة المتوازنة (`Threshold = 0.35`)
**الفكرة ببساطة:**
لاحظ في الخطوة السابقة أن الجملة رقم 3 ("You can create chains...") انفصلت لوحدها لأن تشابهها كان 0.4487 وهو أقل من 0.7.
لو خفضنا العتبة لتكون 0.35، فكل جمل الذكاء الاصطناعي ستندمج معاً في Chunk واحد، وكل جمل فرنسا ستندمج معاً في Chunk ثانٍ!

In [11]:
# run with balanced threshold = 0.35
chunks_balanced = semantic_chunk_text(sentences, embeddings, threshold=0.35)

print(f"📦 Chunks with Threshold = 0.35 (Total: {len(chunks_balanced)}):\n")
for idx, c in enumerate(chunks_balanced, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{c}\n")

📦 Chunks with Threshold = 0.35 (Total: 2):

--- Chunk [1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

--- Chunk [2] ---
The Eiffel Tower is located in Paris. France is a popular tourist destination.



In [12]:
# ✍️ مساحتك للتطبيق العملي (جرب عتبات مختلفة مثل 0.2 أو 0.5 ولاحظ الفرق في عدد القطع هنا):
